## Modularizing PySpark Code
- You want to focus on modularizing code so you can unit test specific blocks
- Makes the code more reusable and clearer to read

### BAD EXAMPLE 
- Blocks of code are across one or two cells
- They are not reusable so code duplication could occur

In [0]:
# Load data
df = (
    spark.read.csv(
        "health.csv", 
        header=True, 
        inferSchema=True
    )
)

# Create new column
df = {
    df.withColumn("NewColumn",
        when(col("Column") == "Value", "NewValue")
        .otherwise(col("Unknown"))
    )
}

## GOOD EXAMPLE
- You can test each function independently
- You can reuse the same function

In [0]:
def load_data(file_path):
    return (
        spark.read.csv(
            file_path, 
            header=True, 
            inferSchema=True
        )
    )

In [0]:
def add_new_col(df, new, s_col):
    return (df
            .withColumn(new,
                        when col(s_col) == '0', 'Normal')
                        .otherwise('Unknown'))

## PROJECT CODE

In [0]:
# Read the test csv file

spark.sql(f'''
          SELECT *
          FROM text.`/Volumes/tdp_bvt_raw/testcsv/testvolume/TestAscent/results.csv`
          ''').display()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType, TimestampType, FloatType
from pyspark.sql.functions import col, when, current_timestamp

In [0]:
csv_schema = StructType([
    StructField("CallGuid", StringType(), True),
    StructField("callstartdate", DateType(), True),
    StructField("enterdatetime", TimestampType(), True),
    StructField("icmestimatedwaittime", IntegerType(), True),
    StructField("EventType", StringType(), True),
    StructField("eventdatetime", TimestampType(), True),
    StructField("Cause", StringType(), True)
])

In [0]:
# Set the csv path
csv_path = "/Volumes/tdp_bvt_raw/testcsv/testvolume/TestAscent/results.csv"
# Read the csv file
ascent_raw = (spark
      .read
      .format('csv')
      .schema(csv_schema)
      .option('header', 'true')
      .load(csv_path)
      # You can access metadata in the file by calling _metadata struct object
      .select(
          "*",
          "_metadata.file_name",
          "_metadata.file_modification_time",
          "_metadata",
          current_timestamp().alias("processing_time")
      )
)

In [0]:
ascent_raw.display()